# Loan Risk Prediction — Full ML Pipeline

End-to-end pipeline: EDA → preprocessing → classical ML → deep learning → evaluation & model persistence.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import joblib
import os

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, precision_recall_curve
)

# ── Reproducibility (FIX: ANN results were non-deterministic)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

OUTPUT_DIR = "saved_models"
os.makedirs(OUTPUT_DIR, exist_ok=True)


## 1. Data Loading & Exploratory Analysis

In [ ]:
def load_and_explore(filepath="loan_risk_prediction_dataset.csv"):
    print("Loading data...")
    df = pd.read_csv(filepath)

    print(f"Raw shape: {df.shape}")
    print(f"\nDuplicate rows: {df.duplicated().sum()}")
    df.drop_duplicates(inplace=True)

    # FIX: inspect missing values before dropping
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if missing.empty:
        print("No missing values found.")
    else:
        print("\nMissing values per column:")
        print(missing.to_string())
        pct = missing / len(df) * 100
        heavy = pct[pct >= 5].index.tolist()
        safe  = pct[pct <  5].index.tolist()
        if heavy:
            print(f"WARNING: high-missing columns — consider imputation: {heavy}")
        df.dropna(subset=safe, inplace=True)

    print(f"Clean shape: {df.shape}")

    # ── Visualisations
    plt.figure(figsize=(16, 5))

    plt.subplot(1, 3, 1)
    df["LoanApproved"].value_counts().plot.pie(
        autopct="%1.1f%%", colors=["#ff9999", "#66b3ff"],
        labels=["Rejected", "Approved"]
    )
    plt.title("Loan approval imbalance")
    plt.ylabel("")

    plt.subplot(1, 3, 2)
    sns.scatterplot(data=df, x="CreditScore", y="LoanAmount",
                    hue="LoanApproved", alpha=0.5, palette="Set1")
    plt.title("Credit score vs loan amount")

    plt.subplot(1, 3, 3)
    num_cols = df.select_dtypes(include="number").columns
    sns.heatmap(df[num_cols].corr(), annot=True, cmap="coolwarm",
                fmt=".2f", linewidths=0.5, square=True)
    plt.title("Numerical correlations")

    plt.tight_layout()
    plt.show()
    return df


## 2. Outlier Removal (train set only)

In [ ]:
def remove_outliers_train(X_train, y_train, num_cols):
    """
    IQR-based outlier removal, train set only.

    FIX: original code filtered column-by-column in a loop, compounding
    row loss each iteration.  We build ONE combined boolean mask across
    all numeric columns and apply it in a single filter step.
    """
    train_data = pd.concat([X_train, y_train], axis=1)
    mask = pd.Series(True, index=train_data.index)

    for col in num_cols:
        Q1 = train_data[col].quantile(0.25)
        Q3 = train_data[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        mask &= (train_data[col] >= lower) & (train_data[col] <= upper)

    n_removed = (~mask).sum()
    print(f"  Outliers removed: {n_removed} rows "
          f"({n_removed / len(train_data) * 100:.1f} %)")

    clean = train_data[mask]
    return clean.drop("LoanApproved", axis=1), clean["LoanApproved"]


## 3. Preprocessing

In [ ]:
def preprocess_data(df, target="LoanApproved"):
    """
    Split first → outlier removal (train only) → scale & encode.

    FIX: numeric columns are auto-detected from the training split
    instead of being hardcoded, preventing KeyError on any dataset variant.
    FIX: preprocessor is saved to disk for inference reuse.
    """
    print("\nPreprocessing data...")

    X = df.drop(target, axis=1)
    y = df[target]

    # 1. Split FIRST — prevents leakage from test set into preprocessing
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )

    # 2. Auto-detect column types from training split (FIX: was hardcoded)
    num_cols = X_train.select_dtypes(include="number").columns.tolist()
    cat_cols = X_train.select_dtypes(include="object").columns.tolist()
    print(f"  Numeric columns    : {num_cols}")
    print(f"  Categorical columns: {cat_cols}")

    # 3. Outlier removal — train set only
    X_train_clean, y_train_clean = remove_outliers_train(
        X_train, y_train, num_cols
    )

    # 4. Fit preprocessor on cleaned train, transform both splits
    preprocessor = ColumnTransformer(transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(
            drop="first", handle_unknown="ignore", sparse_output=False
        ), cat_cols),
    ], remainder="passthrough")

    X_train_proc = preprocessor.fit_transform(X_train_clean)
    X_test_proc  = preprocessor.transform(X_test)

    # 5. Persist preprocessor (FIX: was never saved)
    joblib.dump(preprocessor, os.path.join(OUTPUT_DIR, "preprocessor.pkl"))
    print(f"  Preprocessor saved to {OUTPUT_DIR}/preprocessor.pkl")

    return X_train_proc, X_test_proc, y_train_clean, y_test, preprocessor


## 4. Classical ML Models

In [ ]:
def train_ml_models(X_train, y_train):
    """Train LR, RF, and XGBoost with stratified 5-fold CV."""
    print("\nTraining ML models with cross-validation...")

    neg = (y_train == 0).sum()
    pos = (y_train == 1).sum()
    scale_weight = neg / pos

    models = {
        "Logistic Regression": LogisticRegression(
            class_weight="balanced", random_state=RANDOM_STATE,
            max_iter=1000),
        "Random Forest": RandomForestClassifier(
            class_weight="balanced", n_estimators=100,
            random_state=RANDOM_STATE),
        "XGBoost": XGBClassifier(
            scale_pos_weight=scale_weight, eval_metric="logloss",
            random_state=RANDOM_STATE, verbosity=0),
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True,
                         random_state=RANDOM_STATE)

    trained = {}
    print("--- 5-Fold Stratified CV (F1) ---")
    for name, model in models.items():
        scores = cross_val_score(model, X_train, y_train,
                                 cv=cv, scoring="f1")
        print(f"  {name:25s}: {scores.mean():.4f}  (+/- {scores.std()*2:.4f})")
        model.fit(X_train, y_train)
        # Persist model (FIX: was never saved)
        fname = name.lower().replace(" ", "_") + ".pkl"
        joblib.dump(model, os.path.join(OUTPUT_DIR, fname))
        trained[name] = model

    return trained


## 5. Feature Importance

*(FIX: entirely absent in the original — added for interpretability)*

In [ ]:
def plot_feature_importance(trained_models, preprocessor, top_n=15):
    tree_models = {k: v for k, v in trained_models.items()
                   if hasattr(v, "feature_importances_")}
    if not tree_models:
        print("No tree-based models found.")
        return

    try:
        feature_names = preprocessor.get_feature_names_out()
    except Exception:
        feature_names = None

    n = len(tree_models)
    fig, axes = plt.subplots(1, n, figsize=(7 * n, 5))
    if n == 1:
        axes = [axes]

    for ax, (name, model) in zip(axes, tree_models.items()):
        imp = model.feature_importances_
        if feature_names is not None and len(feature_names) == len(imp):
            fi = pd.Series(imp, index=feature_names)
        else:
            fi = pd.Series(imp)
        fi.nlargest(top_n).sort_values().plot.barh(
            ax=ax, color="steelblue", edgecolor="white"
        )
        ax.set_title(f"Feature importance — {name}")
        ax.set_xlabel("Importance score")
        ax.tick_params(labelsize=9)

    plt.tight_layout()
    plt.show()


## 6. Deep Learning (ANN) & Full Evaluation

In [ ]:
def _find_best_threshold(model, X_val, y_val):
    """
    Find the probability threshold that maximises F1 on a validation slice.
    FIX: original hardcoded 0.5, which is rarely optimal for imbalanced data.
    """
    probs = model.predict(X_val, verbose=0).flatten()
    precisions, recalls, thresholds = precision_recall_curve(y_val, probs)
    with np.errstate(invalid="ignore"):
        f1s = np.where(
            (precisions + recalls) == 0, 0,
            2 * precisions * recalls / (precisions + recalls)
        )
    best_idx = int(np.argmax(f1s[:-1]))
    best_thr = float(thresholds[best_idx])
    print(f"  Best ANN threshold: {best_thr:.3f}  "
          f"(val F1 = {f1s[best_idx]:.4f})")
    return best_thr


In [ ]:
def build_and_train_ann(X_train, y_train):
    """
    FIX 1: replaced deprecated input_shape= kwarg with explicit Input layer.
    FIX 2: threshold tuned on a stratified held-out validation slice.
    FIX 3: model persisted to disk after training.
    """
    neg = (y_train == 0).sum()
    pos = (y_train == 1).sum()
    class_weights = {0: 1.0, 1: float(neg / pos)}

    # Stratified val split for threshold calibration
    idx_tr, idx_val = train_test_split(
        np.arange(len(y_train)), test_size=0.15,
        random_state=RANDOM_STATE, stratify=y_train
    )
    X_tr  = X_train[idx_tr];  y_tr  = np.array(y_train)[idx_tr]
    X_val = X_train[idx_val]; y_val = np.array(y_train)[idx_val]

    ann = Sequential([
        Input(shape=(X_train.shape[1],)),  # FIX: explicit Input layer
        Dense(64, activation="relu"),
        Dropout(0.3),
        Dense(32, activation="relu"),
        Dropout(0.2),
        Dense(1, activation="sigmoid"),
    ])
    ann.compile(optimizer="adam", loss="binary_crossentropy",
                metrics=["accuracy"])

    early_stop = EarlyStopping(
        monitor="val_loss", patience=5,
        restore_best_weights=True, verbose=0
    )

    print("\nTraining ANN with early stopping...")
    ann.fit(
        X_tr, y_tr,
        epochs=100, batch_size=32,
        validation_data=(X_val, y_val),
        callbacks=[early_stop],
        class_weight=class_weights,
        verbose=0,
    )

    best_threshold = _find_best_threshold(ann, X_val, y_val)

    # Persist ANN (FIX: was never saved)
    ann.save(os.path.join(OUTPUT_DIR, "ann_model.keras"))
    print(f"  ANN saved to {OUTPUT_DIR}/ann_model.keras")

    return ann, best_threshold


In [ ]:
def evaluate_all(X_train, y_train, X_test, y_test, ml_models):
    """
    Train ANN, then evaluate all models.
    FIX: added Precision, Recall, ROC-AUC, confusion matrices, ROC curves.
    """
    ann, ann_threshold = build_and_train_ann(X_train, y_train)

    def _row(name, y_true, y_pred, y_prob=None):
        return {
            "Model"    : name,
            "Accuracy" : accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred, zero_division=0),
            "Recall"   : recall_score(y_true, y_pred, zero_division=0),
            "F1"       : f1_score(y_true, y_pred, zero_division=0),
            "ROC-AUC"  : roc_auc_score(y_true, y_prob)
                         if y_prob is not None else float("nan"),
        }

    results = []
    all_preds = {}
    all_probs = {}

    for name, model in ml_models.items():
        preds = model.predict(X_test)
        probs = (model.predict_proba(X_test)[:, 1]
                 if hasattr(model, "predict_proba") else None)
        results.append(_row(name, y_test, preds, probs))
        all_preds[name] = preds
        if probs is not None:
            all_probs[name] = probs

    ann_probs = ann.predict(X_test, verbose=0).flatten()
    ann_preds = (ann_probs >= ann_threshold).astype(int)
    results.append(_row("Deep Learning (ANN)", y_test, ann_preds, ann_probs))
    all_preds["Deep Learning (ANN)"] = ann_preds
    all_probs["Deep Learning (ANN)"] = ann_probs

    # ── Results table
    final_df = (pd.DataFrame(results)
                  .sort_values("F1", ascending=False)
                  .reset_index(drop=True))
    print("\n=== FINAL TEST SET COMPARISON ===")
    print(final_df.to_markdown(index=False, floatfmt=".4f"))

    # ── Confusion matrices
    n = len(all_preds)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
    if n == 1:
        axes = [axes]
    for ax, (name, preds) in zip(axes, all_preds.items()):
        ConfusionMatrixDisplay(
            confusion_matrix(y_test, preds),
            display_labels=["Rejected", "Approved"]
        ).plot(ax=ax, colorbar=False, cmap="Blues")
        ax.set_title(name, fontsize=10)
    plt.suptitle("Confusion matrices — test set", fontsize=12)
    plt.tight_layout()
    plt.show()

    # ── ROC curves
    plt.figure(figsize=(7, 5))
    for name, probs in all_probs.items():
        auc = roc_auc_score(y_test, probs)
        fpr, tpr, _ = roc_curve(y_test, probs)
        plt.plot(fpr, tpr, label=f"{name}  (AUC = {auc:.3f})")
    plt.plot([0, 1], [0, 1], "k--", linewidth=0.8)
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.title("ROC curves — all models")
    plt.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

    return final_df, ann


## 7. Run the Full Pipeline

In [ ]:
if __name__ == "__main__":
    # 1. Load & EDA
    df_clean = load_and_explore()

    # 2. Preprocess
    X_train_final, X_test_final, y_train_final, y_test_final, preprocessor = \
        preprocess_data(df_clean)

    # 3. Classical ML
    trained_ml = train_ml_models(X_train_final, y_train_final)

    # 4. Feature importance
    plot_feature_importance(trained_ml, preprocessor)

    # 5. ANN training + full evaluation
    final_results, ann_model = evaluate_all(
        X_train_final, y_train_final,
        X_test_final,  y_test_final,
        trained_ml
    )
